<p style="text-align: center">
<img src="../../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# 5 — Passing

In this task your Duckiebot must overtake obstacles on the road by pulling into the opposing lane, passing, and merging back safely.

## Objectives

1. **Stationary obstacle** — Duckiebot passes a stationary obstacle (duckiebot or duck).
2. **Lane change maneuver** — Pulls out into the opposing lane, passes, and merges back into lane.
3. **Moving obstacle** — Duckiebot passes a moving duckiebot that travels at a constant (slow) speed. Measures that speed.

## Simulation map

Passing uses the **free drive** map (`passing.tscn`) — same layout as introduction, without traffic signs.

```bash
.venv\Scripts\activate
python launch.py --sim --task passing
```

## Drive modes

| Mode | How |
|------|-----|
| **Manual** | Arrow keys or WASD (default) |
| **Automatic** | Click *Switch to Auto* → *Start* — bot follows lane lines on its own |

## How visual lane servoing works

The automatic mode reuses the same pipeline as the **visual lane servoing** task:

1. **Lane detection** — Convert the camera image to HSV and threshold yellow (left line) and white (right line) masks.
2. **Slice sampling** — Scan horizontal strips in the lower part of the image; take the mean x-position of each line in each slice.
3. **Lateral error** — Compare the lane center to the image center. If both lines are visible, use their midpoint; if only one line is visible, estimate the center using a learned half-lane width.
4. **PD controller** — `steering = p_gain × error + d_gain × Δerror`, clipped to `max_steer`.
5. **Motor mixing** — `left = speed − steering`, `right = speed + steering`. Slow down on curves; stop if too few lane pixels are detected.
6. **Apply** — Send left/right PWM to the wheel driver every frame.

Config lives in `config/lane_servoing_config.yaml`. Passing-specific logic (overtake, merge) goes in `tasks/passing/packages/agent.py`.

## Implementation

Start from `PassingAgent` in `tasks/passing/packages/agent.py` — it currently extends the lane servoing agent. Add passing behavior on top of autonomous lane following.